# Phase 2: Feature Engineering (Pandas Version)

This notebook reproduces the PySpark feature engineering pipeline using **pandas** only.

Sections:
1. **Session Patterns** — 6 features per user per week
2. **Engagement Decay** — 6 features per user per week
3. **Streaming Quality** — 8 features per user per week
4. **Final Join** — merge all sections into `weekly_features`

## Data Loading

In [1]:
import pandas as pd
import numpy as np
import datetime

DATA_DIR = "../data/raw"

users = pd.read_parquet(f"{DATA_DIR}/users.parquet")
sessions = pd.read_parquet(f"{DATA_DIR}/session_logs.parquet")
games = pd.read_parquet(f"{DATA_DIR}/game_catalog.parquet")
sub_events = pd.read_parquet(f"{DATA_DIR}/subscription_events.parquet")
payments = pd.read_parquet(f"{DATA_DIR}/payments.parquet")

print(f"users: {len(users):,} rows")
print(f"sessions: {len(sessions):,} rows")
print(f"games: {len(games):,} rows")
print(f"sub_events: {len(sub_events):,} rows")
print(f"payments: {len(payments):,} rows")

users: 50,000 rows
sessions: 3,480,097 rows
games: 200 rows
sub_events: 38,538 rows
payments: 239,048 rows


## Temporal Constants

In [2]:
DATA_START    = datetime.date(2024, 1, 1)    # Week 1 start
BASELINE_END  = datetime.date(2024, 1, 29)   # Week 5 start = baseline end
OBS_START     = datetime.date(2024, 1, 29)   # Observation window start (week 5)
OBS_END       = datetime.date(2024, 2, 26)   # Observation window end (week 9 start, exclusive)
BASELINE_WEEKS = 4

## Shared Base: `obs_sessions`

Filter sessions to observation window and compute `week_num` (1–4) and `duration_min`.

**PySpark equivalent:**
```python
obs_sessions = sessions.filter(
    (F.col("start_time").cast("date") >= F.lit(OBS_START)) &
    (F.col("start_time").cast("date") < F.lit(OBS_END))
).withColumn(
    "week_num",
    (F.datediff(F.col("start_time").cast("date"), F.lit(OBS_START)) / 7).cast("int") + 1
).withColumn(
    "duration_min",
    (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60.0
)
```

**Pandas translation:**
- `F.col("start_time").cast("date")` → `pd.to_datetime(sessions["start_time"]).dt.date`
- `F.datediff(a, b)` → `(a - b).dt.days`
- `F.unix_timestamp("end_time") - F.unix_timestamp("start_time")` → `(end_time - start_time).dt.total_seconds()`

In [ ]:
# Ensure datetime types
sessions["start_time"] = pd.to_datetime(sessions["start_time"])
sessions["end_time"] = pd.to_datetime(sessions["end_time"])

# Filter to observation window
obs_sessions = sessions[
    (sessions["start_time"].dt.date >= OBS_START) &
    (sessions["start_time"].dt.date < OBS_END)
].copy()

# week_num: days since OBS_START // 7 + 1 → gives weeks 1-4
obs_sessions["week_num"] = (
    (obs_sessions["start_time"].dt.date - OBS_START).apply(lambda d: d.days) // 7 + 1
)

# duration in minutes
obs_sessions["duration_min"] = (
    (obs_sessions["end_time"] - obs_sessions["start_time"]).dt.total_seconds() / 60.0
)

print(f"obs_sessions: {len(obs_sessions):,} rows")
print(f"week_num range: {obs_sessions['week_num'].min()} – {obs_sessions['week_num'].max()}")

## Section 1: Session Patterns (6 features)

Features per user per week:
- `weekly_session_count` — number of sessions
- `avg_session_duration_min` — mean session length
- `total_playtime_min` — sum of session lengths
- `peak_hour_ratio` — fraction of sessions during 19:00–23:00
- `weekend_ratio` — fraction of sessions on Sat/Sun
- `session_regularity` — std of inter-session gaps (minutes)

**Key PySpark → Pandas translations:**

| PySpark | Pandas |
|---------|--------|
| `.groupBy("user_id", "week_num").agg(F.count(...))` | `.groupby(["user_id", "week_num"]).agg(...)` |
| `F.avg(F.when(condition, 1).otherwise(0))` | `.apply(lambda x: condition.mean())` or vectorized `.mean()` |
| `F.lag(...).over(Window.partitionBy(...).orderBy(...))` | `.sort_values(...).groupby(...).shift(1)` |
| `F.stddev(...)` | `.std()` |

In [ ]:
# ══════════════════════════════════════════════
# Section 1: Session Patterns
# Output: session_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# Helper columns for peak hours and weekends
obs_sessions["is_peak_hour"] = obs_sessions["start_time"].dt.hour.between(19, 23).astype(int)
# dayofweek: Monday=0 ... Sunday=6; weekend = 5 (Sat) or 6 (Sun)
# PySpark dayofweek: Sunday=1, Saturday=7; .isin(1,7) = weekend
# Pandas equivalent: dayofweek in [5, 6]
obs_sessions["is_weekend"] = obs_sessions["start_time"].dt.dayofweek.isin([5, 6]).astype(int)

# Core aggregations
session_patterns = obs_sessions.groupby(["user_id", "week_num"]).agg(
    weekly_session_count=("session_id", "count"),
    avg_session_duration_min=("duration_min", "mean"),
    total_playtime_min=("duration_min", "sum"),
    peak_hour_ratio=("is_peak_hour", "mean"),
    weekend_ratio=("is_weekend", "mean"),
).reset_index()

print(f"session_patterns: {len(session_patterns):,} rows")

In [ ]:
# Session regularity: std of inter-session gaps within each (user, week)
#
# PySpark used Window + lag to compute prev_end, then stddev.
# Pandas: sort → groupby shift → compute gap → groupby std.

sorted_sessions = obs_sessions.sort_values(["user_id", "week_num", "start_time"])

# Lag end_time within (user_id, week_num)
sorted_sessions["prev_end"] = sorted_sessions.groupby(["user_id", "week_num"])["end_time"].shift(1)

# Inter-session gap in minutes
sorted_sessions["inter_session_gap_min"] = (
    (sorted_sessions["start_time"] - sorted_sessions["prev_end"]).dt.total_seconds() / 60.0
)

# Std of gaps per user per week (only where gap is not null)
session_regularity = (
    sorted_sessions.dropna(subset=["inter_session_gap_min"])
    .groupby(["user_id", "week_num"])["inter_session_gap_min"]
    .std()
    .reset_index()
    .rename(columns={"inter_session_gap_min": "session_regularity"})
)

# Combine
session_features = session_patterns.merge(
    session_regularity, on=["user_id", "week_num"], how="left"
)
session_features["session_regularity"] = session_features["session_regularity"].fillna(0)

print(f"session_features: {len(session_features):,} rows")
session_features.head()

### Section 1 Validation

In [ ]:
session_features.merge(users[["user_id", "persona"]], on="user_id").groupby("persona").agg(
    avg_sessions=("weekly_session_count", "mean"),
    avg_duration=("avg_session_duration_min", "mean"),
    avg_regularity=("session_regularity", "mean"),
    avg_peak=("peak_hour_ratio", "mean"),
    avg_weekend=("weekend_ratio", "mean"),
).round(2)

## Section 2: Engagement Decay (6 features)

Features per user per week:
- `session_count_wow_change` — week-over-week % change in session count
- `playtime_wow_change` — week-over-week % change in playtime
- `session_count_vs_baseline` — ratio vs baseline period average
- `playtime_vs_baseline` — ratio vs baseline period average
- `longest_inactive_days` — longest streak of consecutive days with no sessions

**Key translations:**

| PySpark | Pandas |
|---------|--------|
| `crossJoin(spark.createDataFrame(...))` | `pd.MultiIndex.from_product(...)` |
| `F.lag(...).over(Window.partitionBy("user_id").orderBy("week_num"))` | `.groupby("user_id").shift(1)` |
| `spark.sql("SELECT explode(sequence(...))")` | `pd.date_range(...)` |
| `F.sum(...).over(Window.partitionBy("user_id").orderBy("date"))` | `.groupby("user_id").cumsum()` |

In [ ]:
# ══════════════════════════════════════════════
# Section 2: Engagement Decay
# Output: engagement_features (user_id, week_num, 6 features)
# ══════════════════════════════════════════════

# --- Scaffold: ensure every user has all 4 weeks ---
# PySpark: crossJoin to create all (user_id, week_num) combos
# Pandas: MultiIndex.from_product → reindex

obs_user_ids = obs_sessions["user_id"].unique()
week_nums = [1, 2, 3, 4]

all_user_weeks = pd.DataFrame(
    [(uid, w) for uid in obs_user_ids for w in week_nums],
    columns=["user_id", "week_num"]
)

fill_cols = [
    "weekly_session_count", "total_playtime_min",
    "avg_session_duration_min", "peak_hour_ratio",
    "weekend_ratio", "session_regularity",
]

session_features_filled = all_user_weeks.merge(
    session_features, on=["user_id", "week_num"], how="left"
)
session_features_filled[fill_cols] = session_features_filled[fill_cols].fillna(0)

print(f"session_features_filled: {len(session_features_filled):,} rows")

In [ ]:
# --- 2a. Week-over-week change ---
# PySpark: Window + lag, then manual % change formula
# Pandas: sort + groupby shift, then vectorized division

sff = session_features_filled.sort_values(["user_id", "week_num"]).copy()

sff["prev_session_count"] = sff.groupby("user_id")["weekly_session_count"].shift(1)
sff["prev_playtime"] = sff.groupby("user_id")["total_playtime_min"].shift(1)

# % change: (current - prev) / prev; None when prev is null or 0
def safe_pct_change(current, previous):
    result = np.where(
        previous.isna() | (previous == 0),
        np.nan,
        (current - previous) / previous
    )
    return result

wow_features = sff[["user_id", "week_num"]].copy()
wow_features["session_count_wow_change"] = safe_pct_change(
    sff["weekly_session_count"], sff["prev_session_count"]
)
wow_features["playtime_wow_change"] = safe_pct_change(
    sff["total_playtime_min"], sff["prev_playtime"]
)

print(f"wow_features: {len(wow_features):,} rows")
wow_features.head(8)

In [ ]:
# --- 2b. Baseline comparison ---
# Filter sessions to baseline period (weeks 1-4), compute per-user averages

baseline_sessions = sessions[
    (sessions["start_time"].dt.date >= DATA_START) &
    (sessions["start_time"].dt.date < BASELINE_END)
].copy()

baseline_sessions["duration_min"] = (
    (baseline_sessions["end_time"] - baseline_sessions["start_time"]).dt.total_seconds() / 60.0
)

user_baseline = baseline_sessions.groupby("user_id").agg(
    baseline_session_count=("session_id", "count"),
    baseline_total_playtime=("duration_min", "sum"),
).reset_index()

user_baseline["baseline_avg_session_count"] = user_baseline["baseline_session_count"] / BASELINE_WEEKS
user_baseline["baseline_avg_playtime"] = user_baseline["baseline_total_playtime"] / BASELINE_WEEKS

# Join with filled session features and compute ratios
baseline_ratios = session_features_filled[["user_id", "week_num", "weekly_session_count", "total_playtime_min"]].merge(
    user_baseline[["user_id", "baseline_avg_session_count", "baseline_avg_playtime"]],
    on="user_id", how="left"
)

baseline_features = baseline_ratios[["user_id", "week_num"]].copy()
baseline_features["session_count_vs_baseline"] = np.where(
    baseline_ratios["baseline_avg_session_count"].isna() | (baseline_ratios["baseline_avg_session_count"] == 0),
    np.nan,
    baseline_ratios["weekly_session_count"] / baseline_ratios["baseline_avg_session_count"]
)
baseline_features["playtime_vs_baseline"] = np.where(
    baseline_ratios["baseline_avg_playtime"].isna() | (baseline_ratios["baseline_avg_playtime"] == 0),
    np.nan,
    baseline_ratios["total_playtime_min"] / baseline_ratios["baseline_avg_playtime"]
)

print(f"baseline_features: {len(baseline_features):,} rows")

In [ ]:
# --- 2c. Longest inactive days ---
# PySpark: explode(sequence(...)) to build date scaffold, then cumsum trick
# Pandas: pd.date_range → cross join → cumsum trick (same logic)

# Build date scaffold for obs window
all_dates = pd.DataFrame({
    "date": pd.date_range(start=OBS_START, end=pd.Timestamp(OBS_END) - pd.Timedelta(days=1), freq="D").date
})
all_dates["week_num"] = [(d - OBS_START).days // 7 + 1 for d in all_dates["date"]]

# Cross join: every user × every date
user_date_scaffold = pd.DataFrame({"user_id": obs_user_ids}).merge(all_dates, how="cross")

# Dates that had sessions
session_dates = obs_sessions[["user_id", "start_time"]].copy()
session_dates["date"] = session_dates["start_time"].dt.date
session_dates = session_dates[["user_id", "date"]].drop_duplicates()
session_dates["had_session"] = 1

# Mark active days
daily_activity = user_date_scaffold.merge(
    session_dates, on=["user_id", "date"], how="left"
)
daily_activity["had_session"] = daily_activity["had_session"].fillna(0).astype(int)

# Cumsum trick: group inactive streaks
daily_activity = daily_activity.sort_values(["user_id", "date"])
daily_activity["session_cumsum"] = daily_activity.groupby("user_id")["had_session"].cumsum()

# For inactive days, compute streak length within each (user, cumsum group)
inactive = daily_activity[daily_activity["had_session"] == 0].copy()
inactive["streak_len"] = inactive.groupby(["user_id", "session_cumsum"]).cumcount() + 1

# Longest streak per user per week
longest_inactive = (
    inactive.groupby(["user_id", "week_num"])["streak_len"]
    .max()
    .reset_index()
    .rename(columns={"streak_len": "longest_inactive_days"})
)

print(f"longest_inactive: {len(longest_inactive):,} rows")

In [ ]:
# --- 2d. Combine all Section 2 features ---
engagement_features = (
    wow_features
    .merge(baseline_features, on=["user_id", "week_num"], how="outer")
    .merge(longest_inactive, on=["user_id", "week_num"], how="outer")
)
engagement_features["longest_inactive_days"] = engagement_features["longest_inactive_days"].fillna(0)

print(f"engagement_features: {len(engagement_features):,} rows")
engagement_features.head()

### Section 2 Validation

In [ ]:
engagement_features.merge(users[["user_id", "persona"]], on="user_id").groupby("persona").agg(
    avg_wow_change=("session_count_wow_change", "mean"),
    avg_vs_baseline=("session_count_vs_baseline", "mean"),
    avg_play_vs_base=("playtime_vs_baseline", "mean"),
    avg_inactive=("longest_inactive_days", "mean"),
).round(3)

## Section 3: Streaming Quality (8 features)

Features per user per week:
- `avg_latency`, `avg_fps`, `frame_drop_rate`, `disconnect_rate`
- `avg_bitrate`, `avg_jitter`, `packet_loss_avg`, `crash_exit_ratio`

This section is straightforward — all features are simple `groupby` + `mean` aggregations.

| PySpark | Pandas |
|---------|--------|
| `F.avg(F.when(F.col("exit_type").isin(...), 1).otherwise(0))` | Create boolean column → `.mean()` |

In [ ]:
# ══════════════════════════════════════════════
# Section 3: Streaming Quality
# Output: streaming_features (user_id, week_num, 8 features)
# ══════════════════════════════════════════════

obs_sessions["is_crash_exit"] = obs_sessions["exit_type"].isin(
    ["crash", "disconnect", "timeout"]
).astype(int)

streaming_features = obs_sessions.groupby(["user_id", "week_num"]).agg(
    avg_latency=("avg_latency_ms", "mean"),
    avg_fps=("avg_fps", "mean"),
    frame_drop_rate=("total_frame_drops", "mean"),
    disconnect_rate=("disconnect_count", "mean"),
    avg_bitrate=("avg_bitrate_mbps", "mean"),
    avg_jitter=("avg_jitter_ms", "mean"),
    packet_loss_avg=("packet_loss_rate", "mean"),
    crash_exit_ratio=("is_crash_exit", "mean"),
).reset_index()

print(f"streaming_features: {len(streaming_features):,} rows")
streaming_features.head()

### Section 3 Validation

In [ ]:
streaming_features.merge(users[["user_id", "persona"]], on="user_id").groupby("persona").agg(
    avg_latency=("avg_latency", "mean"),
    avg_fps=("avg_fps", "mean"),
    crash_exit_ratio=("crash_exit_ratio", "mean"),
).round(2)

## Final Join

Merge all feature sections into a single `weekly_features` DataFrame.

| PySpark | Pandas |
|---------|--------|
| `.join(..., on=[...], how="outer")` | `.merge(..., on=[...], how="outer")` |

In [ ]:
# ══════════════════════════════════════════════
# Final: Merge all feature sections
# ══════════════════════════════════════════════

weekly_features = (
    session_features_filled
    .merge(engagement_features, on=["user_id", "week_num"], how="outer")
    .merge(streaming_features, on=["user_id", "week_num"], how="outer")
)

print(f"weekly_features: {len(weekly_features):,} rows, {len(weekly_features.columns)} cols")
print(f"Columns: {list(weekly_features.columns)}")

### Final Validation

In [ ]:
total_users = len(users)
print(f"Total users: {total_users:,}")
print(f"weekly_features rows: {len(weekly_features):,}")
print(f"Unique users in weekly_features: {weekly_features['user_id'].nunique():,}")
print(f"Week distribution:\n{weekly_features['week_num'].value_counts().sort_index()}")

In [ ]:
# Persona behavior profiles
persona_stats = weekly_features.merge(
    users[["user_id", "persona", "subscription_tier"]], on="user_id"
)

persona_stats.groupby("persona").agg(
    avg_sessions=("weekly_session_count", "mean"),
    avg_duration=("avg_session_duration_min", "mean"),
    avg_playtime=("total_playtime_min", "mean"),
    avg_wow_change=("session_count_wow_change", "mean"),
    avg_vs_baseline=("session_count_vs_baseline", "mean"),
    avg_inactive=("longest_inactive_days", "mean"),
    avg_latency=("avg_latency", "mean"),
    avg_crash_exit=("crash_exit_ratio", "mean"),
).round(3)

In [ ]:
# Churn decay progression: about_to_churn users should show week-over-week decline
churn_users = persona_stats[persona_stats["persona"] == "about_to_churn"]

churn_weekly_trend = churn_users.groupby("week_num").agg(
    sessions=("weekly_session_count", "mean"),
    duration=("avg_session_duration_min", "mean"),
    playtime=("total_playtime_min", "mean"),
    vs_baseline=("session_count_vs_baseline", "mean"),
    inactive_days=("longest_inactive_days", "mean"),
    latency=("avg_latency", "mean"),
    crash_ratio=("crash_exit_ratio", "mean"),
).round(3)

print("about_to_churn weekly trend (should show decay):")
churn_weekly_trend

In [ ]:
# Contrast: hardcore users should be stable
hardcore_users = persona_stats[persona_stats["persona"] == "hardcore"]

hardcore_weekly_trend = hardcore_users.groupby("week_num").agg(
    sessions=("weekly_session_count", "mean"),
    duration=("avg_session_duration_min", "mean"),
    playtime=("total_playtime_min", "mean"),
    vs_baseline=("session_count_vs_baseline", "mean"),
    inactive_days=("longest_inactive_days", "mean"),
    latency=("avg_latency", "mean"),
    crash_ratio=("crash_exit_ratio", "mean"),
).round(3)

print("hardcore weekly trend (should be stable):")
hardcore_weekly_trend